In [46]:
import hashlib
import os
from typing import List

In [47]:
def prf(seed: bytes, counter: int, outlen=32) -> bytes:
    return hashlib.shake_256(seed + counter.to_bytes(4, 'big')).digest(outlen)

In [48]:
def aggregate(values: List[bytes]) -> bytes:
    result = bytearray(values[0])
    for v in values[1:]:
        for i in range(len(result)):
            result[i] ^= v[i]
    return bytes(result)

In [49]:
def dilithium_commit(y: bytes) -> bytes:
    return hashlib.sha256(b"commit" + y).digest()

def dilithium_decompose(w: bytes):
    # Split bytes into "high" and "low"
    mid = len(w) // 2
    return w[:mid], w[mid:]

def dilithium_challenge(w1H: bytes, message: bytes) -> bytes:
    return hashlib.sha256(w1H + message).digest()

def dilithium_response(y: bytes, c: bytes, sk_share: bytes) -> bytes:
    return hashlib.sha256(b"resp" + y + c + sk_share).digest()

def dilithium_recompute(z: bytes, c: bytes, pk: bytes) -> bytes:
    return hashlib.sha256(b"recomp" + z + c + pk).digest()

def dilithium_make_hint(w1L: bytes, w1p: bytes) -> bytes:
    return hashlib.sha256(b"hint" + w1L + w1p).digest()[:8]

def dilithium_verify(pk: bytes, msg: bytes, sig: bytes) -> bool:
    # Placeholder "verification"
    return len(sig) > 0

In [50]:
NUM_ICS = 3
MESSAGE = b"Hello from Myst + Masked Dilithium"
SIGN_COUNTER = 1

# Simulated public key
public_key = hashlib.sha256(b"public-key").digest()

# Simulated secret key split
def split_secret(secret: bytes, n: int):
    shares = [os.urandom(len(secret)) for _ in range(n - 1)]
    last = bytearray(secret)
    for s in shares:
        for i in range(len(last)):
            last[i] ^= s[i]
    shares.append(bytes(last))
    return shares

master_secret = hashlib.sha256(b"secret-key").digest()
secret_shares = split_secret(master_secret, NUM_ICS)
prf_seeds = [os.urandom(32) for _ in range(NUM_ICS)]

In [51]:
y_vals = []
w1_parts = []

for i in range(NUM_ICS):
    y_i = prf(prf_seeds[i], SIGN_COUNTER)
    w1_i = dilithium_commit(y_i)
    y_vals.append(y_i)
    w1_parts.append(w1_i)

# print(y_vals, w1_parts)
print("✓ IC commitments generated")

✓ IC commitments generated


In [52]:
w1 = aggregate(w1_parts)
w1H, w1L = dilithium_decompose(w1)
challenge = dilithium_challenge(w1H, MESSAGE)

# print(challenge)
print("✓ Host computed challenge")

✓ Host computed challenge


In [53]:
z_parts = []

for i in range(NUM_ICS):
    z_i = dilithium_response(y_vals[i], challenge, secret_shares[i])
    z_parts.append(z_i)

# print(z_parts)
print("✓ IC masked responses generated")


✓ IC masked responses generated


In [57]:
z = aggregate(z_parts)
w1p = dilithium_recompute(z, challenge, public_key)
hint = dilithium_make_hint(w1L, w1p)

# print(z)
print("✓ Host generated hint")

✓ Host generated hint


In [55]:
signature = z + challenge + hint
print("✓ Signature generated")

✓ Signature generated


In [56]:
valid = dilithium_verify(public_key, MESSAGE, signature)
print("Signature valid:", valid)

Signature valid: True
